In [1]:
import sqlite3
import pandas as pd

# create a SQLite database in memory and load our three tables into it
conn = sqlite3.connect(":memory:")
for name in ["sales", "customers", "products"]:
    pd.read_csv(f"{name}.csv").to_sql(name, conn, index=False, if_exists="replace")

# a small helper so we can run SQL and see the result as a nice table
def run(sql):
    return pd.read_sql_query(sql, conn)

print("database ready with 3 tables: sales, customers, products")

database ready with 3 tables: sales, customers, products


***TASK 01 Twenty queries of increasing difficulty***

• 1. Show the first 10 rows of the sales table.

In [2]:
run("SELECT * FROM sales LIMIT 10")

,order_id,date,customer_id,product_id,quantity
0,O00001,2024-10-31,C999,P019,2
1,O00002,2024-03-14,C999,P014,1
2,O00003,2024-10-21,C999,P004,1
3,O00004,2024-01-03,C999,P003,3
4,O00005,2024-10-18,C999,P018,4
5,O00006,2024-10-15,C010,P006,2
6,O00007,2024-10-12,C013,P016,5
7,O00008,2024-08-31,C046,P003,2
8,O00009,2024-06-21,C048,P006,5
9,O00010,2024-09-15,C003,P006,2


• 2. Count the total number of orders.

In [5]:
run("SELECT COUNT() FROM sales")

,COUNT()
0,1000


• 3. Show all orders with a quantity of 5.

In [7]:
run("SELECT order_id , quantity from sales WHERE quantity = 5")

,order_id,quantity
0,O00007,5
1,O00009,5
2,O00014,5
3,O00020,5
4,O00022,5
...,...,...
186,O00981,5
187,O00988,5
188,O00989,5
189,O00990,5


• 4. List all products in the 'Electronics' category.

In [9]:
run("SELECT product_name from products WHERE category = 'Electronics'")

,product_name
0,Laptop
1,Mouse
2,Keyboard
3,Monitor
4,Webcam
5,Headphones
6,USB Hub
7,Cable Set
8,Power Bank
9,Speaker


• 5. Show the 5 most expensive products (by unit price).

In [11]:
run("SELECT product_name, unit_price  from products ORDER BY unit_price DESC LIMIT 5")

,product_name,unit_price
0,Laptop,800
1,Standing Desk,400
2,Tablet,350
3,Monitor,250
4,Chair,180


• 6. Count how many customers are in each region.

In [13]:
run("""
SELECT region , COUNT(*) AS num_of_customers
FROM customers
GROUP BY region
""")

,region,num_of_customers
0,East,13
1,North,12
2,South,11
3,West,16


• 7. Find the total quantity sold for each product.

In [18]:
run("""SELECT product_id ,
    SUM(quantity) AS quantity_sold
    FROM sales
    GROUP BY product_id
""")

,product_id,quantity_sold
0,P001,126
1,P002,138
2,P003,159
3,P004,177
4,P005,171
5,P006,145
6,P007,159
7,P008,178
8,P009,156
9,P010,172


• 8. Find the average unit price per category.

In [25]:
run("""SELECT category ,
    AVG(unit_price) AS Avg_price

    FROM products
    GROUP BY category


""")

,category,Avg_price
0,Accessories,31.000000
1,Electronics,163.750000
2,Office,206.666667
3,Stationery,11.500000


• 9. Count orders per product, but only show products with more than 50 orders (use HAVING).

In [26]:
run("""
    SELECT product_id, COUNT(*) AS orders
    FROM sales
    GROUP BY product_id
    HAVING COUNT(*) > 50
    ORDER BY orders DESC
""")

,product_id,orders
0,P004,62
1,P008,59
2,P019,57
3,P003,56
4,P010,54
5,P005,54
6,P020,53
7,P007,53
8,P017,51


• 10. Find the single highest and lowest unit price in the products table.

In [37]:
run("""SELECT 
    MAX(unit_price) AS highest_unit_price,
    MIN(unit_price) AS lowest_unit_price
FROM products;

""")


,highest_unit_price,lowest_unit_price
0,800,8


• 11. Join sales to products and show each order's product name and price.

In [39]:
run("""
    SELECT s.order_id, s.product_id, p.product_name, p.unit_price, s.quantity
    FROM sales s
    JOIN products p ON s.product_id = p.product_id
    
""")

,order_id,product_id,product_name,unit_price,quantity
0,O00001,P019,Microphone,110,2
1,O00002,P014,Water Bottle,20,1
2,O00003,P004,Monitor,250,1
3,O00004,P003,Keyboard,45,3
4,O00005,P018,Speaker,90,4
...,...,...,...,...,...
995,O00996,P016,Cable Set,30,3
996,O00997,P004,Monitor,250,3
997,O00998,P007,USB Hub,35,1
998,O00999,P004,Monitor,250,4


• 12. Compute total revenue (quantity times price) across all sales.

In [40]:
run("""
    SELECT SUM(s.quantity * p.unit_price) AS total_revenue
    FROM sales s
    JOIN products p ON s.product_id = p.product_id
""")

,total_revenue
0,405150


13. Compute total revenue per region (join all three tables).

In [41]:
run("""
    SELECT c.region,
           SUM(s.quantity * p.unit_price) AS revenue
    FROM sales s
    JOIN customers c ON s.customer_id = c.customer_id
    JOIN products  p ON s.product_id  = p.product_id
    GROUP BY c.region
    ORDER BY revenue DESC
""")

,region,revenue
0,East,121636
1,West,117271
2,North,83444
3,South,81814


• 14. Compute total revenue per product category.

In [42]:
run("""
    SELECT p.category,
           SUM(s.quantity * p.unit_price) AS revenue
    FROM sales s
        JOIN products  p ON s.product_id  = p.product_id
    GROUP BY p.category
    ORDER BY revenue DESC
""")

,category,revenue
0,Electronics,285690
1,Office,104000
2,Accessories,12081
3,Stationery,3379


• 15. Find how many sales reference a customer who isn't in the customers table (hint: LEFT JOIN + IS NULL).

In [43]:
run("""
    SELECT COUNT(*) AS unknown_customer_sales
    FROM sales s
    LEFT JOIN customers c
        ON s.customer_id = c.customer_id
    WHERE c.customer_id IS NULL
""")


,unknown_customer_sales
0,5


In [47]:
run("""
    SELECT COUNT(*) AS customers_never_ordered
    FROM customers c
    LEFT JOIN sales s
        ON c.customer_id = s.customer_id
    WHERE s.order_id IS NULL
""")

,customers_never_ordered
0,2


• 16. Rank all products by total revenue using a window function.

In [48]:
run("""
    SELECT product_id, revenue,
           RANK() OVER (ORDER BY revenue DESC) AS revenue_rank
    FROM (
        SELECT s.product_id, SUM(s.quantity * p.unit_price) AS revenue
        FROM sales s JOIN products p ON s.product_id = p.product_id
        GROUP BY s.product_id
    )
    ORDER BY revenue_rank
    
""")

,product_id,revenue,revenue_rank
0,P001,100800,1
1,P010,68800,2
2,P020,54600,3
3,P004,44250,4
4,P009,28080,5
5,P019,18590,6
6,P006,17400,7
7,P018,11520,8
8,P005,10260,9
9,P017,8050,10


• 17. Compute a running total of revenue by month.

In [ ]:
run("""
    WITH monthly_revenue AS (
        SELECT
            strftime('%Y-%m', s.date) AS month,
            SUM(s.quantity * p.unit_price) AS revenue
        FROM sales s
        JOIN products p
            ON s.product_id = p.product_id
        GROUP BY strftime('%Y-%m', s.date)
    wqqqqqq53
    SELECT
        month,
        revenue,
        SUM(revenue) OVER (ORDER BY month) AS running_total
    FROM monthly_revenue
    ORDER BY month
""")

,month,revenue,running_total
0,2024-01,32561,32561
1,2024-02,47995,80556
2,2024-03,38608,119164
3,2024-04,28702,147866
4,2024-05,27291,175157
5,2024-06,31720,206877
6,2024-07,34522,241399
7,2024-08,31247,272646
8,2024-09,29860,302506
9,2024-10,30620,333126


• 18. Find the top-selling product within each category (PARTITION BY).

In [50]:
run("""
    WITH product_sales AS (
        SELECT
            p.category,
            p.product_name,
            SUM(s.quantity) AS total_sold
        FROM products p
        JOIN sales s
            ON p.product_id = s.product_id
        GROUP BY p.category, p.product_name
    ),
    ranked_products AS (
        SELECT
            category,
            product_name,
            total_sold,
            ROW_NUMBER() OVER (
                PARTITION BY category
                ORDER BY total_sold DESC
            ) AS rank
        FROM product_sales
    )
    SELECT
        category,
        product_name,
        total_sold
    FROM ranked_products
    WHERE rank = 1
    ORDER BY category
""")

,category,product_name,total_sold
0,Accessories,Phone Stand,137
1,Electronics,Monitor,177
2,Office,Desk Lamp,178
3,Stationery,Notebook,158


• 19. Using a CTE, find all products that sold above the average product revenue.

In [51]:
run("""
    WITH product_revenue AS (
        SELECT
            p.product_id,
            p.product_name,
            SUM(s.quantity * p.unit_price) AS total_revenue
        FROM products p
        JOIN sales s
            ON p.product_id = s.product_id
        GROUP BY p.product_id, p.product_name
    )
    SELECT
        product_id,
        product_name,
        total_revenue
    FROM product_revenue
    WHERE total_revenue > (
        SELECT AVG(total_revenue)
        FROM product_revenue
    )
    ORDER BY total_revenue DESC
""")

,product_id,product_name,total_revenue
0,P001,Laptop,100800
1,P010,Standing Desk,68800
2,P020,Tablet,54600
3,P004,Monitor,44250
4,P009,Chair,28080


• 20. Compute each region's revenue as a percentage of total revenue.

In [52]:
run("""
    WITH region_revenue AS (
        SELECT
            c.region,
            SUM(s.quantity * p.unit_price) AS revenue
        FROM sales s
        JOIN customers c
            ON s.customer_id = c.customer_id
        JOIN products p
            ON s.product_id = p.product_id
        GROUP BY c.region
    )
    SELECT
        region,
        revenue,
        ROUND(
            revenue * 100.0 / (SELECT SUM(revenue) FROM region_revenue),
            2
        ) AS revenue_percentage
    FROM region_revenue
    ORDER BY revenue_percentage DESC
""")

,region,revenue,revenue_percentage
0,East,121636,30.10
1,West,117271,29.02
2,North,83444,20.65
3,South,81814,20.24


***TASK 02 Side-by-side: Pandas vs SQL***

• Pick one non-trivial analysis question — for example, 'total revenue per region per month', or 'the top 3 products
by revenue in each category'.

• Solve it fully in Pandas (merge, groupby, etc.) — the way you did yesterday.

• Solve the exact same question in SQL (joins, GROUP BY, maybe a window function).

• Print both results and confirm the numbers match.

• Write a short paragraph comparing the two: which was easier to write? Which was easier to read? When would
you choose each?

In [54]:
import pandas as pd 

sales = pd.read_csv("sales.csv")
customers = pd.read_csv("customers.csv")
products = pd.read_csv("products.csv")

print (f"sales Shape  {sales.shape}  ")
print (f"customers Shape  {customers.shape}  ")
print (f"products Shape  {products.shape}  ")



sales Shape  (1000, 5)  
customers Shape  (52, 4)  
products Shape  (20, 4)  


In [62]:
# Revenue per region per month
pandas_result = (
    sales
    .merge(customers, on="customer_id", how="inner")
    .merge(products, on="product_id", how="inner")
)

pandas_result["revenue"] = (
    pandas_result["quantity"] * pandas_result["unit_price"]
)

pandas_result["month"] = pd.to_datetime(
    pandas_result["date"]
).dt.to_period("M")

pandas_result = (
    pandas_result
    .groupby(["region", "month"])["revenue"]
    .sum()
    .reset_index()
    .sort_values(["region", "month"])
)

print(pandas_result.head())

  region    month  revenue
0   East  2024-01    11506
1   East  2024-02    11110
2   East  2024-03    13911
3   East  2024-04    10275
4   East  2024-05     5976


In [63]:
sql_result = run("""
    SELECT
        c.region,
        strftime('%Y-%m', s.date) AS month,
        SUM(s.quantity * p.unit_price) AS revenue
    FROM sales s
    JOIN customers c
        ON s.customer_id = c.customer_id
    JOIN products p
        ON s.product_id = p.product_id
    GROUP BY
        c.region,
        strftime('%Y-%m', s.date)
    ORDER BY
        c.region,
        month
    LIMIT 5
""")

print(sql_result)

  region    month  revenue
0   East  2024-01    11506
1   East  2024-02    11110
2   East  2024-03    13911
3   East  2024-04    10275
4   East  2024-05     5976


In [61]:

pandas_result["month"] = pandas_result["month"].astype(str)
sql_result["month"] = sql_result["month"].astype(str)

print(
    pandas_result.equals(sql_result)

)

True


In [66]:
'''pandas is the easier to write and read
    for database we have to always use the sql one 
    pandas is easier in every situation 
'''

'pandas is the easier to write and read\n    for database we have to always use the sql one \n    pandas is easier in every situation \n'

***TASK 03 Window functions practice***

• Compute a running total of revenue over the whole year, ordered by date (a cumulative sum).

In [67]:
run("""
    SELECT
        s.date,
        s.quantity * p.unit_price AS revenue,
        SUM(s.quantity * p.unit_price) OVER (
            ORDER BY s.date
        ) AS running_revenue
    FROM sales s
    JOIN products p
        ON s.product_id = p.product_id
    ORDER BY s.date
""")

,date,revenue,running_revenue
0,2024-01-02,1250,2654
1,2024-01-02,180,2654
2,2024-01-02,1200,2654
3,2024-01-02,24,2654
4,2024-01-03,135,5757
...,...,...,...
995,2024-12-30,54,401060
996,2024-12-30,1000,401060
997,2024-12-30,800,401060
998,2024-12-31,4000,405150


• Rank customers by their total spending, showing the top 10.

In [68]:
run("""
    SELECT
        c.customer_id,
        c.customer_name,
        SUM(s.quantity * p.unit_price) AS total_spending,
        RANK() OVER (
            ORDER BY SUM(s.quantity * p.unit_price) DESC
        ) AS spending_rank
    FROM customers c
    JOIN sales s
        ON c.customer_id = s.customer_id
    JOIN products p
        ON s.product_id = p.product_id
    GROUP BY
        c.customer_id,
        c.customer_name
    ORDER BY total_spending DESC
    LIMIT 10
""")

,customer_id,customer_name,total_spending,spending_rank
0,C024,Customer_24,19300,1
1,C015,Customer_15,17480,2
2,C003,Customer_3,12885,3
3,C039,Customer_39,12568,4
4,C023,Customer_23,12464,5
5,C043,Customer_43,12214,6
6,C027,Customer_27,11775,7
7,C050,Customer_50,11644,8
8,C007,Customer_7,11114,9
9,C029,Customer_29,10887,10


• For each region, rank its customers by spending using PARTITION BY, so ranking restarts per region.

In [69]:
run("""
    SELECT
        c.region,
        c.customer_id,
        c.customer_name,
        SUM(s.quantity * p.unit_price) AS total_spending,
        RANK() OVER (
            PARTITION BY c.region
            ORDER BY SUM(s.quantity * p.unit_price) DESC
        ) AS spending_rank
    FROM customers c
    JOIN sales s
        ON c.customer_id = s.customer_id
    JOIN products p
        ON s.product_id = p.product_id
    GROUP BY
        c.region,
        c.customer_id,
        c.customer_name
    ORDER BY
        c.region,
        spending_rank
""")

,region,customer_id,customer_name,total_spending,spending_rank
0,East,C015,Customer_15,17480,1
1,East,C003,Customer_3,12885,2
2,East,C043,Customer_43,12214,3
3,East,C029,Customer_29,10887,4
4,East,C034,Customer_34,10843,5
5,East,C049,Customer_49,9920,6
6,East,C021,Customer_21,9745,7
7,East,C008,Customer_8,8986,8
8,East,C017,Customer_17,5967,9
9,East,C011,Customer_11,5905,10


• Compute, for each month, that month's revenue AND the previous month's revenue side by side (look up the
LAG window function).

In [70]:
run("""
    WITH monthly_revenue AS (
        SELECT
            strftime('%Y-%m', s.date) AS month,
            SUM(s.quantity * p.unit_price) AS revenue
        FROM sales s
        JOIN products p
            ON s.product_id = p.product_id
        GROUP BY strftime('%Y-%m', s.date)
    )
    SELECT
        month,
        revenue,
        LAG(revenue) OVER (
            ORDER BY month
        ) AS previous_month_revenue
    FROM monthly_revenue
    ORDER BY month
""")

,month,revenue,previous_month_revenue
0,2024-01,32561,NaN
1,2024-02,47995,32561.0
2,2024-03,38608,47995.0
3,2024-04,28702,38608.0
4,2024-05,27291,28702.0
5,2024-06,31720,27291.0
6,2024-07,34522,31720.0
7,2024-08,31247,34522.0
8,2024-09,29860,31247.0
9,2024-10,30620,29860.0


• Using LAG, compute month-over-month growth: the percentage change in revenue from one month to the next.

In [71]:
run("""
    WITH monthly_revenue AS (
        SELECT
            strftime('%Y-%m', s.date) AS month,
            SUM(s.quantity * p.unit_price) AS revenue
        FROM sales s
        JOIN products p
            ON s.product_id = p.product_id
        GROUP BY strftime('%Y-%m', s.date)
    ),
    monthly_with_previous AS (
        SELECT
            month,
            revenue,
            LAG(revenue) OVER (
                ORDER BY month
            ) AS previous_revenue
        FROM monthly_revenue
    )
    SELECT
        month,
        revenue,
        previous_revenue,
        ROUND(
            (revenue - previous_revenue) * 100.0
            / previous_revenue,
            2
        ) AS mom_growth_percent
    FROM monthly_with_previous
    ORDER BY month
""")

,month,revenue,previous_revenue,mom_growth_percent
0,2024-01,32561,NaN,NaN
1,2024-02,47995,32561.0,47.40
2,2024-03,38608,47995.0,-19.56
3,2024-04,28702,38608.0,-25.66
4,2024-05,27291,28702.0,-4.92
5,2024-06,31720,27291.0,16.23
6,2024-07,34522,31720.0,8.83
7,2024-08,31247,34522.0,-9.49
8,2024-09,29860,31247.0,-4.44
9,2024-10,30620,29860.0,2.55
